# LMA Phase 2: TELUGU Transformer Pretraining

Training using real model code imported from bundle.

In [ ]:
# ===== CONFIG-ONLY CELL: Modify these before running =====
ROOT_DIR = "/kaggle/input/lma-telugu-phase2"  # Dataset bundle path
OUT_DIR = "/kaggle/working/checkpoints"   # Save checkpoints here
CHECK_DIR = None                          # Resume from checkpoint (if available)

# Hyperparameters: None means use config JSON defaults
hp = dict(
    batch_size=None,
    learning_rate=None,
    total_steps=None,
    warmup_steps=None,
    weight_decay=None,
    eval_steps=None,
    checkpoint_steps=None,
    amp=True,
)


In [ ]:
import sys
import os
import argparse
from pathlib import Path

# Setup path to import from bundle
sys.path.insert(0, ROOT_DIR)
os.chdir('/kaggle/working')  # For checkpoint/log output

# Install tokenizers if needed
import subprocess
subprocess.run(['pip', 'install', 'tokenizers'], capture_output=True)

print(f'Root: {ROOT_DIR}')
print(f'Output: {OUT_DIR}')
print(f'Resume: {CHECK_DIR}')


In [ ]:
# Import the real training code (no reimplementation)
from train.train import main

print('✅ Imported training code from bundle')


In [ ]:
# Build command-line arguments by mimicking train.py's argparse
# Filter out None hyperparams (use config JSON defaults)
args_dict = {k: v for k, v in hp.items() if v is not None}

# Handle resume: compute resume_from path
resume_from = None
if CHECK_DIR and Path(CHECK_DIR).exists():
    potential_ckpt = Path(CHECK_DIR) / 'checkpoint_last.pt'
    if potential_ckpt.exists():
        resume_from = str(potential_ckpt)
        print(f'Will resume from: {resume_from}')

# Create args namespace
args = argparse.Namespace(
    batch_size=args_dict.get('batch_size', None),
    learning_rate=args_dict.get('learning_rate', None),
    total_steps=args_dict.get('total_steps', None),
    warmup_steps=args_dict.get('warmup_steps', None),
    weight_decay=args_dict.get('weight_decay', None),
    eval_steps=args_dict.get('eval_steps', None),
    checkpoint_steps=args_dict.get('checkpoint_steps', None),
    amp=args_dict.get('amp', True),
    resume_from=resume_from,
)

print('✅ Arguments prepared')


In [ ]:
# Run training with real Trainer class (checkpointing, logging, AMP, etc. all included)
print('\n' + '='*60)
print(f'Starting {language.upper()} training...')
print('='*60 + '\n')

main(args)

print('\n' + '='*60)
print('✅ Training complete!')
print('='*60)


In [ ]:
# Final summary
import glob

checkpoints = sorted(glob.glob(f'{OUT_DIR}/checkpoint_*.pt'))
logs = sorted(glob.glob('logs/*.log'))

print(f'\n📊 Training outputs:')
if checkpoints:
    print(f'  Checkpoints:')
    for ckpt in checkpoints:
        size_mb = Path(ckpt).stat().st_size / 1e6
        print(f'    {ckpt} ({size_mb:.1f} MB)')

if logs:
    print(f'  Logs:')
    for log in logs:
        print(f'    {log}')

print(f'\n📝 To resume in next run:')
print(f'  1. Save this notebook output as a Kaggle dataset')
print(f'  2. Set CHECK_DIR = "/kaggle/input/<this-output-dataset>/checkpoints"')
print(f'  3. Run the notebook again')
